# Experiment notebook for AgentCompromiseLab
# Section 1: Introduction

"""
Goal: Load results from `results/bypass_metrics.csv` and `results/controlled_metrics.csv`,
compute detection rates per mode and per payload family, and visualize the improvement
from baseline heuristics to the validator-augmented pipeline.
"""


# Section 2: Install dependencies (run this cell if needed)
# !pip install -r requirements.txt


In [ ]:
# Section 3: Imports
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set(style="whitegrid")

print('pandas', pd.__version__)
print('matplotlib', plt.__version__)


In [ ]:
# Section 4: Helper functions
from typing import Tuple, Dict


def detection_rate_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """Compute detection rates grouped by mode and family."""
    total = df.groupby(['mode', 'family']).size().rename('total')
    blocked = df[df['status']=='blocked'].groupby(['mode','family']).size().rename('blocked')
    merged = pd.concat([total, blocked], axis=1).fillna(0)
    merged['detection_rate'] = merged['blocked'] / merged['total']
    return merged.reset_index()


def plot_detection_rates(df_summary: pd.DataFrame, out_path: str = 'results/fig_detection_rates.png') -> None:
    plt.figure(figsize=(8,5))
    pivot = df_summary.pivot(index='family', columns='mode', values='detection_rate')
    pivot.plot(kind='bar', rot=0)
    plt.title('Detection Rate by Payload Family (baseline vs validator)')
    plt.ylabel('Detection rate')
    plt.ylim(0,1)
    plt.tight_layout()
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path)
    plt.show()


In [ ]:
# Section 5: Load and preview data
bypass_path = 'results/bypass_metrics.csv'
controlled_path = 'results/controlled_metrics.csv'

if Path(bypass_path).exists():
    df_bypass = pd.read_csv(bypass_path)
    display(df_bypass.head())
else:
    print('Missing', bypass_path)

if Path(controlled_path).exists():
    df_controlled = pd.read_csv(controlled_path)
    display(df_controlled.head())
else:
    print('Missing', controlled_path)


In [ ]:
# Section 6: Compute summaries and show tables
if 'df_bypass' in globals():
    summary_bypass = detection_rate_from_df(df_bypass)
    display(summary_bypass)
else:
    print('No bypass data loaded')

if 'df_controlled' in globals():
    summary_controlled = detection_rate_from_df(df_controlled)
    display(summary_controlled)
else:
    print('No controlled data loaded')


In [ ]:
# Section 7: Plot detection rates (controlled experiment)
if 'summary_controlled' in globals():
    plot_detection_rates(summary_controlled, out_path='results/fig_detection_rates_controlled.png')
else:
    print('No controlled summary available for plotting')


In [ ]:
# Section 8: Show examples that bypassed baseline but blocked by validator
if 'df_bypass' in globals():
    # aggregate to find payloads where baseline had any safe but validator blocked
    baseline_safe = df_bypass[(df_bypass['mode']=='baseline') & (df_bypass['status']!='blocked')]['payload'].unique()
    validator_blocked = df_bypass[(df_bypass['mode']=='validator') & (df_bypass['status']=='blocked')]['payload'].unique()
    examples = [p for p in baseline_safe if p in validator_blocked]
    print('Found', len(examples), 'payload texts that bypass baseline but blocked by validator. Showing up to 5 examples:')
    for p in examples[:5]:
        print('-', p)
else:
    print('No bypass dataset loaded')


In [ ]:
# Section 9: Save a small CSV summary for the README or external analysis
if 'summary_controlled' in globals():
    summary_controlled.to_csv('results/summary_controlled.csv', index=False)
    print('Wrote results/summary_controlled.csv')
else:
    print('No summary to save')


# Section 10: Interview talking points
"""
- Baseline heuristics were bypassed by obfuscation and subtle phrasing (detection ~33% on our synthetic bypass set).
- Adding an independent validator raised detection to 100% on that synthetic set (see figure results/fig_detection_rates_controlled.png).
- For interviews: show the notebook plots, the sample bypass payloads, and explain the next steps to scale and harden the approach.
"""
